This week we will be looking at fundamentals of working with AI agents. First, we will look at the LLM client (here we use OpenAI). 

## Setting up the OpenAI client

The OpenAI client requires an API key and the base url of the model provider:

In [1]:
from openai import OpenAI
from atlas.vault import get_secret

MODEL_NAME = "deepseek-flash"
BASE_URL = "https://api.deepseek.com"
API_KEY = get_secret("DEEPSEEK_API_KEY")
client = OpenAI(api_key=API_KEY, base_url=BASE_URL)

### Our first LLM call

Recall that behind the API is a language model that performs text completion. Here we choose a specific model under `model` then the **message chain** is passed to that model as prompt. The first message is expected to be the **system prompt** with role `system`. Then, we can append our messages which has role `user`. 

Next, we have further configuration like whether we stream content or not, and whether the model will use **reasoning** to respond along with a *guidance* on how much **reasoning effort** the model should use in its response (generally, the higher the reasoning, the better).

In [8]:
response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "system", "content": "You are a helpful assistant"},
        {"role": "user", "content": "Tell me a fun fact."},
    ],
    stream=False,
    reasoning_effort="high",
    extra_body={"thinking": {"type": "enabled"}},
)

print(response.choices[0].message.content)


A fun fact: **Sharks are older than trees.** Sharks have been around for roughly 450 million years, while the first trees appeared about 385 million years ago—so sharks were swimming around long before anything tree-like existed on land. 🦈🌳


Notice that the output is markdown, so let's render it:

In [9]:
from IPython.display import display, Markdown
display(Markdown("> " + response.choices[0].message.content))

> A fun fact: **Sharks are older than trees.** Sharks have been around for roughly 450 million years, while the first trees appeared about 385 million years ago—so sharks were swimming around long before anything tree-like existed on land. 🦈🌳

For testing we can set `n > 0` in `client.chat.completions.create` to get a sample of `n` responses for the same prompt. But by default `n=1`. Hence, `response.choices` is a singleton and `[0]` just gets the sole response. Let's extract the reasoning trace:

In [10]:
from rich import print_json

print_json(response.choices[0].model_dump_json())
display(Markdown("> " + response.choices[0].model_dump()["message"]["reasoning_content"]))

{
  "finish_reason": "stop",
  "index": 0,
  "logprobs": null,
  "message": {
    "content": "A fun fact: **Sharks are older than trees.** Sharks have been around for roughly 450 million years, while the first trees appeared about 385 million years ago—so sharks were swimming around long before anything tree-like existed on land. 🦈🌳",
    "refusal": null,
    "role": "assistant",
    "annotations": null,
    "audio": null,
    "function_call": null,
    "tool_calls": null,
    "reasoning_content": "We need answer user asks fun fact. Need provide fun fact. Simple. Could choose interesting. Need no tools. Ensure accurate. Fun fact: Honey never spoils? Archaeologists found edible honey in ancient Egyptian tombs; due low moisture, acidity, hydrogen peroxide. This is common but maybe oversimplified. Or octopuses have three hearts and blue blood. Fun fact: Wombat cube poop. Need concise. \"A group of flamingos is called a flamboyance.\" Fun. Provide one fun fact. Maybe surprising: \"Sharks existed before trees.\" That's true-ish: sharks ~450 million years, trees ~385 million. Fun. Or \"Bananas are berries, strawberries aren't.\" Good. Need answer. Final."
  }
}

> We need answer user asks fun fact. Need provide fun fact. Simple. Could choose interesting. Need no tools. Ensure accurate. Fun fact: Honey never spoils? Archaeologists found edible honey in ancient Egyptian tombs; due low moisture, acidity, hydrogen peroxide. This is common but maybe oversimplified. Or octopuses have three hearts and blue blood. Fun fact: Wombat cube poop. Need concise. "A group of flamingos is called a flamboyance." Fun. Provide one fun fact. Maybe surprising: "Sharks existed before trees." That's true-ish: sharks ~450 million years, trees ~385 million. Fun. Or "Bananas are berries, strawberries aren't." Good. Need answer. Final.

Here we also see the role of the model is `assistant`.

The following examples repeat the same request and display steps, so let's put them in a function. `chat` accepts the message history and returns the full response for chaining. Its keyword arguments expose the model, reasoning effort and thinking mode; `single_paragraph` only changes how the answer is displayed.


In [18]:
def chat(
    messages,
    *,
    model=MODEL_NAME,
    reasoning_effort="high",
    thinking_type="enabled"
):
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        stream=False,
        reasoning_effort=reasoning_effort,
        extra_body={"thinking": {"type": thinking_type}},
    )

    # expose reasoning trace and response content as markdown
    message = response.choices[0].message
    reasoning = message.model_dump().get("reasoning_content").replace("\n", " ")
    if reasoning:
        print("Reasoning:")
        display(Markdown("> " + reasoning))

    content = message.content or ""
    content = content.replace("\n", " ")
    print("Response:")
    display(Markdown("> " + content))
    return response

### Chaining messages

Chaining messages allows us to basically converse with the LLM. Let's make the same message as above:

In [19]:
SYSTEM_PROMPT = "You are a helpful assistant"
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "Tell me a fun fact."},
]

# First call
response = chat(messages)

Reasoning:


> We need answer fun fact. Need concise. Pick interesting. Need no tools. Maybe "octopuses have three hearts and blue blood" but common. Fun fact about wombat cube poop. Or "bananas are berries, strawberries aren't" common. Need fun. Could say: "A group of flamingos is called a flamboyance." Or "There are more possible chess games than atoms in observable universe" maybe. Let's craft one fun fact with brief explanation. Maybe "Wombats produce cube-shaped poop—their intestines have varying elasticity that molds feces into cubes, helping it stay put as territory markers." That's fun. Answer.

Response:


> Fun fact: Wombats poop cubes. Their intestines have uneven elasticity that molds the poop into cube shapes—handy for marking territory because the cubes don’t roll away.

Let's create a user response and chain it to the message history:

In [20]:
follow_up = "That's cool!"

msg = response.choices[0].message
messages.append({"role": msg.role, "content": msg.content})
messages.append({"role": "user", "content": follow_up})

# Second call
response = chat(messages)


Reasoning:


> We need to respond to user's enthusiasm. They said "That's cool!" after wombat fact. Need likely continue friendly, maybe add another fun fact or acknowledge. Keep concise. Could say glad you liked it, and offer another related fun fact? Desired oververbosity 5, but simple. Maybe respond with another fun fact about wombats or animals. Should not overdo. We can say "Right? Nature is weirdly efficient. If you want another: wombat pouches face backward..." That's a good follow-up. Keep brief.

Response:


> Right? Nature is weirdly efficient. Bonus wombat fact: their pouches face backward, so when they dig, dirt doesn’t get shoveled into the pouch with the baby.

The response of the model is nice and conversational due to maintaining the message history.

### Templating

Templating lets us reuse the same instructions with different inputs. A Python **f-string** can insert a topic and audience into a prompt. Let's send that prompt using the client we set up earlier:


In [26]:
def explain(topic: str, audience: str, max_words: int = 100):
    SYSTEM_PROMPT = "You are a helpful assistant"
    prompt = (
        f"Explain {topic} to {audience}. "
        f"Use one concrete example and keep the answer under {max_words} words."
    )
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]

    print("Prompt:")
    display(Markdown("> " + prompt))
    return chat(messages)


explain(topic="AI agents", audience="a Python beginner");

Prompt:


> Explain AI agents to a Python beginner. Use one concrete example and keep the answer under 100 words.

Reasoning:


> We need answer under 100 words, explain AI agents to Python beginner with one concrete example. Need concise. Need count words. We can say: AI agent = program that uses LLM to decide steps and call tools until goal. Example: Python script with a "weather" tool; user asks "Should I pack an umbrella in Paris?" Agent decides to call weather API, reads result, then answers. You write tool function and loop; LLM chooses actions. Keep under 100. Count words. Need maybe 70 words. Let's craft exactly. "An AI agent is a program that uses a language model to choose actions—like calling functions or APIs—until it completes a goal. Example: a Python script gives the agent a `get_weather(city)` function. User asks, 'Should I pack an umbrella in Paris?' The agent decides to call `get_weather('Paris')`, reads the result, and answers. You provide tools; the model decides when to use them." Count: An1 AI2 agent3 is4 a5 program6 that7 uses8 a9 language10 model11 to12 choose13 actions—like14 calling15 functions16 or17 APIs—until18 it19 completes20 a21 goal22. Example23: a24 Python25 script26 gives27 the28 agent29 a30 get_weather(city)31 function32. User33 asks34, Should35 I36 pack37 an38 umbrella39 in40 Paris41? The42 agent43 decides44 to45 call46 get_weather('Paris')47, reads48 the49 result50, and51 answers52. You53 provide54 tools55; the56 model57 decides58 when59 to60 use61 them62. Under 100. Good.

Response:


> An AI agent is a program that uses a language model to choose actions—like calling functions or APIs—until it completes a goal. Example: a Python script gives the agent a `get_weather(city)` function. User asks, “Should I pack an umbrella in Paris?” The agent decides to call `get_weather('Paris')`, reads the result, and answers. You provide tools; the model decides when to use them.

That seems like a pretty good explanation of AI agents for beginners. Changing `topic` or `audience` and rerunning the cell gives us a new request with the same instructions. Python fills in `{topic}` and `{audience}` before the API call; the model receives the completed text.


In [27]:
explain(topic="AI agents", audience="a mathematician");

Prompt:


> Explain AI agents to a mathematician. Use one concrete example and keep the answer under 100 words.

Reasoning:


> We need answer to user: explain AI agents to mathematician, one concrete example, under 100 words. Need concise. Mathematician framing. Need define agent as system using LLM? AI agents: perceive state, choose actions via policy, pursue objective, possibly tools/memory. Mathematician: an agent is a policy π mapping observations to actions, coupled to environment, optimizing expected return; LLM provides flexible policy. Concrete example: travel booking agent? Maybe math problem solver? One concrete example under 100 words. Need ensure under 100 words. Count. Draft:  "An AI agent is a policy π: observations → actions, interacting with an environment to maximize expected return. Unlike a single model call, it loops: observe, decide, act, update. Example: a theorem-proving agent receives a goal, proposes a proof step, calls a proof checker, reads errors, and iterates until verified or budget exhausted. Formally, it’s a controlled Markov process; the LLM supplies a flexible, learned policy and tools extend its action space."  Count words: An1 AI2 agent3 is4 a5 policy6 π:7 observations8 →9 actions,10 interacting11 with12 an13 environment14 to15 maximize16 expected17 return.18 Unlike19 a20 single21 model22 call,23 it24 loops:25 observe,26 decide,27 act,28 update.29 Example:30 a31 theorem-proving32 agent33 receives34 a35 goal,36 proposes37 a38 proof39 step,40 calls41 a42 proof43 checker,44 reads45 errors,46 and47 iterates48 until49 verified50 or51 budget52 exhausted.53 Formally,54 it’s55 a56 controlled57 Markov58 process;59 the60 LLM61 supplies62 a63 flexible,64 learned65 policy66 and67 tools68 extend69 its70 action71 space.72 Under 100. Good. Need final only.

Response:


> An AI agent is a policy π: observations → actions, interacting with an environment to maximize expected return. Unlike a single model call, it loops: observe, decide, act, update. Example: a theorem-proving agent receives a goal, proposes a proof step, calls a proof checker, reads errors, and iterates until verified or budget exhausted. Formally, it’s a controlled Markov process; the LLM supplies a flexible learned policy, and tools extend its action space.

That's pretty cool.

:::{.callout-note}
Interestingly, the reasoning model is actually counting. Counting like this using reasoning tokens seems wasteful, esp. with high word count. The alternative is the model doing a **tool call** to bash / Python to write a script for counting the words. This is discussed further below.
:::

## What is an AI agent?

So far, `chat` and `explain` produce answers when we call them. We decide which prompt comes next and when the conversation ends. In contrast, an **AI agent** chooses some of those next steps itself: it works toward a goal, uses the tools available to it, and responds to what it observes. A useful working description is [an LLM with tools in a feedback loop to achieve a goal]{.mark}. The important part is *autonomy*. The model can select an action based on the current state, while the application controls which actions are available and when execution must stop.


### Workflows and agents

[Anthropic's *Building effective agents*](https://www.anthropic.com/research/building-effective-agents) distinguishes two ways of organizing LLM applications:

- In a **workflow**, application code determines the sequence or structure of the steps.
- In an **agent**, the model has more control over the next action and tool choice.

Consider the word limit in our `explain` function. We could always generate a draft, count its words with Python, and ask for a revision if it is too long. That is a workflow: the code prescribes the steps, even though it uses a tool and repeats an LLM call.

An agent has more freedom within its task. For a request to prepare a short explanation grounded in sources, it might decide to search first, inspect a source, draft an answer, or check its length depending on what is missing. Autonomy describes the decisions delegated to the model; it does not require unrestricted access to the environment.


### Five workflow patterns

For many applications, optimizing single LLM calls with retrieval and in-context examples is usually enough. But it can be good to have agents arranged in certain patterns depending on the use case. The examples below apply them to a writing assistant. The patterns below are based on the same Anthropic article. 

| Pattern | How the work is organized | Example |
|:--|:---|:---|
| **Prompt chaining** | Fixed stages pass their results to the next stage, with checks/gates between them. | Create an outline, check its coverage, then draft the explanation. |
| **Routing** | A classifier selects a specialized path. | Route a request to a beginner explanation or a mathematical treatment. |
| **Parallelization** | Independent calls run concurrently and their results are combined. | Review a draft for clarity and factual support in separate calls. |
| **Orchestrator-workers** | A coordinating model determines subtasks and combines worker results. | Break a research question into smaller questions, investigate them, then synthesize an answer. |
| **Evaluator-optimizer** | A candidate draft is evaluated and revised using feedback. | Revise an explanation until it meets a stated rubric or reaches a retry limit. |

These patterns can be combined. The orchestrator-workers pattern already allows dynamic task decomposition inside a defined structure, so the workflow–agent distinction is a useful description of control rather than a rigid classification of every application.


### Function calling

**Function calling** lets an LLM ask the application to run a tool. The application describes the available functions, what they do, and which arguments they accept. For a delivery question, the model might return a structured request naming `lookup_order` and supplying `{"order_id": "A123"}` as its arguments.

Models can learn when to use a tool, which one to choose, and what arguments to supply during **post-training**. For example, the authors of [*Toolformer*](https://arxiv.org/abs/2302.04761) fine-tuned a pretrained language model on text augmented with API calls and their results.


### Acting on feedback

An agent loop connects a model's proposed action to an observation from the environment:

```{mermaid}
flowchart TD
    A["Goal and message history"] --> B["Model chooses the next step"]
    B --> C{"Use a tool?"}
    C -->|Yes| D["Application validates and executes the call"]
    D --> E["Add the result or error to the history"]
    E --> B
    C -->|No| F["Return a final answer"]
```

For example, suppose a support agent receives a report that an order has not arrived. It might look up the order, inspect the tracking result, and then choose between explaining a delay, asking for missing information, or escalating the case. A failed lookup changes what it should do next. The model needs the actual tool result to make that decision.

The model proposes a tool name and arguments; application code performs the operation and returns its result. Writing that an order was checked does not perform a lookup. The environment supplies evidence that the model's generated text alone cannot provide.

The loop also needs stopping rules enforced by the application. It can stop when the task is complete, request human help when blocked, or halt after a limit on steps, time, or spending. A model that keeps requesting actions should not be able to extend its own budget.


### Observability and evaluation

A flexible execution path makes failures harder to diagnose. **Observability** records enough of a run to reconstruct what happened: available messages, tool names and arguments, results, errors, latency, and usage. A trace can show whether a poor answer followed a failed lookup, missing context, or an incorrect interpretation of a valid result.

**Evaluation** asks whether the system accomplished the task. Anthropic's [*Demystifying evals for AI agents*](https://www.anthropic.com/research/demystifying-evals-for-ai-agents) distinguishes the execution trace from the final outcome. For our support example, a convincing message saying the issue was resolved is only one artifact. We would also check the order records, the action actually taken, and whether the customer received an appropriate resolution.

Different checks answer different questions:

- **Code-based checks** verify conditions such as a word limit, required fields, or a change in a test database.
- **Model-based checks**, often called *LLM-as-a-judge*, assess a response against a rubric, such as whether its explanation is supported by the supplied evidence.
- **Human review** handles ambiguous cases and helps calibrate the automated judgments.

A judge can provide feedback inside an evaluator-optimizer workflow, but it can also score completed runs without changing them. Its score is an estimate of quality and needs comparison with human judgments; it does not establish that an external action succeeded.

For a support system, a useful experiment would compare a simple baseline with the proposed agent on the same representative requests. Measure correct resolutions, unsupported claims, appropriate escalations, time, and cost. Repeat runs to expose variation. In production, track whether the system reduces resolution time and repeat contacts while maintaining an acceptable error rate. Fluent answers alone would not demonstrate that improvement.


### Guardrails and execution limits

Giving a model more choices also creates more possible failure paths. It can select the wrong tool, accept an unsupported claim, repeat an unsuccessful action, or spend more than the task warrants. **Guardrails** are checks and restrictions around execution that keep the system within its intended boundaries.

For the support example, those controls could include:

- **Tool permissions:** allow access only to the current customer's records, and validate arguments before an operation runs.
- **Execution budgets:** cap steps, retries, elapsed time, and spending; provide a clear path for escalation.
- **Output checks:** verify required fields and factual support before returning an answer. A valid JSON structure checks the shape of an answer, while its claims still need evidence.
- **Approval boundaries:** require an authorized person to approve actions such as issuing a refund or changing an account.

These controls operate at different points: before a tool call, during the loop, or before an answer is released. An instruction in a prompt can guide behavior; permission checks and limits in application code enforce the allowed actions.

Guardrails also need evaluation. A system that blocks valid requests too often can fail its purpose, while one that misses disallowed actions can exceed its authority. The course develops these controls further in later weeks.


### Start from a measurable problem

The lecture highlights two common design traps. The first is choosing an agent before defining the problem. Its example of a request for a "culture agent" becomes more concrete once the underlying concern is identified as employee attrition. Understanding why people are leaving comes before deciding whether an LLM application would help.

For a support assistant, the problem might be slow resolution of delivery questions. A starting design could use one LLM call with the relevant order information supplied by the application. A more flexible loop becomes worth testing if real cases require different sequences of lookups. Compare both designs against the same success criteria, including their cost and latency.

The second trap is **anthropomorphizing the architecture**: assigning roles such as researcher, writer, and manager, then assuming that a human-like team structure will improve the result. Those roles can help separate prompts or responsibilities, but their names do not establish expertise or independence. Multiple generated answers can repeat the same error.

Add stages or agents to address an observed failure, then measure whether they help. For example, a separate factual review is useful if it catches unsupported claims against trusted records. A reviewer that merely agrees with the draft adds little evidence. Handling plausible but incorrect output is part of designing the system: supply relevant evidence, check consequential claims, and give the application a way to decline or escalate when it cannot establish an answer.


### Using agents and engineering agents

The phrase *agentic engineering* can refer to two different activities: using coding agents to help build software, or building the agent systems themselves. The skills overlap, but the work has a different focus. Using an agent involves specifying tasks and reviewing its changes; engineering an agent involves designing its tools, control loop, evaluations, and execution boundaries. This course focuses on building agents with code.
